# Exploratory Data Analysis - Olist E-Commerce

Analisis eksploratif untuk menjawab pertanyaan bisnis menggunakan kombinasi SQL (aggregasi)
dan Python (visualisasi, segmentasi).

**Prasyarat:** Jalankan `01_load_to_sqlite.ipynb` dan `03_cleaning.ipynb` terlebih dahulu.

## Pertanyaan Bisnis

1. Bagaimana tren pertumbuhan bisnis Olist dari waktu ke waktu?
2. Kategori produk apa yang paling menguntungkan dan bagaimana karakteristiknya?
3. Apa faktor utama yang menyebabkan keterlambatan pengiriman?
4. Bagaimana distribusi geografis pelanggan mempengaruhi biaya dan waktu pengiriman?
5. Bagaimana perilaku pembelian pelanggan dan peluang retensi?

## Persiapan

In [ ]:
import sqlite3

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

DB_PATH = "../data/database/olist.db"
conn = sqlite3.connect(DB_PATH)

---
## 1. Tren Bisnis (Pertanyaan 1)

Melihat perkembangan jumlah order dan total revenue per bulan.

In [ ]:
# SQL: aggregasi order dan revenue per bulan
df_tren = pd.read_sql_query("""
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS bulan,
    COUNT(DISTINCT o.order_id) AS jumlah_order,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(SUM(oi.freight_value), 2) AS total_ongkir
FROM orders o
    JOIN order_item oi ON o.order_id = oi.order_id
GROUP BY bulan
ORDER BY bulan
""", conn)

df_tren.head()

In [ ]:
# Python: visualisasi tren
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(df_tren["bulan"], df_tren["jumlah_order"], marker="o", markersize=4)
ax1.set_ylabel("Jumlah Order")
ax1.set_title("Tren Jumlah Order per Bulan")

ax2.plot(df_tren["bulan"], df_tren["total_revenue"], marker="o", markersize=4, color="tab:green")
ax2.set_ylabel("Revenue (BRL)")
ax2.set_title("Tren Revenue per Bulan")
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 2. Analisis Produk (Pertanyaan 2)

Kategori produk dengan penjualan terbanyak, serta distribusi harga.

In [ ]:
# SQL: top 15 kategori by jumlah terjual
df_kategori = pd.read_sql_query("""
SELECT
    p.product_category_name AS kategori,
    COUNT(*) AS jumlah_terjual,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(AVG(oi.price), 2) AS rata_harga
FROM order_item oi
    JOIN products p ON oi.product_id = p.product_id
GROUP BY kategori
ORDER BY jumlah_terjual DESC
LIMIT 15
""", conn)

df_kategori

In [ ]:
# Python: bar chart top kategori
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

ax1.barh(df_kategori["kategori"][::-1], df_kategori["jumlah_terjual"][::-1])
ax1.set_xlabel("Jumlah Terjual")
ax1.set_title("Top 15 Kategori (by Quantity)")

ax2.barh(df_kategori["kategori"][::-1], df_kategori["total_revenue"][::-1], color="tab:green")
ax2.set_xlabel("Revenue (BRL)")
ax2.set_title("Top 15 Kategori (by Revenue)")
ax2.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.tight_layout()
plt.show()

In [ ]:
# SQL: semua harga untuk distribusi
df_harga = pd.read_sql_query("SELECT price, freight_value FROM order_item", conn)

# Python: distribusi harga
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(df_harga["price"], bins=50, edgecolor="white")
ax1.set_xlabel("Harga (BRL)")
ax1.set_ylabel("Frekuensi")
ax1.set_title("Distribusi Harga Produk")
ax1.set_xlim(0, 500)

ax2.boxplot(df_harga["price"], vert=True)
ax2.set_ylabel("Harga (BRL)")
ax2.set_title("Boxplot Harga Produk")

plt.tight_layout()
plt.show()

print(df_harga["price"].describe())

---
## 3. Analisis Pengiriman (Pertanyaan 3)

Rata-rata waktu pengiriman dan hubungan antara berat produk dengan ongkos kirim.

In [ ]:
# SQL: waktu pengiriman (dalam hari)
df_kirim = pd.read_sql_query("""
SELECT
    ROUND(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp), 1)
        AS hari_kirim,
    ROUND(julianday(order_estimated_delivery_date) - julianday(order_purchase_timestamp), 1)
        AS hari_estimasi
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
""", conn)

df_kirim.head()

In [ ]:
# Python: distribusi waktu kirim vs estimasi
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(df_kirim["hari_kirim"], bins=50, alpha=0.7, label="Aktual", edgecolor="white")
ax.hist(df_kirim["hari_estimasi"], bins=50, alpha=0.5, label="Estimasi", edgecolor="white")
ax.set_xlabel("Hari")
ax.set_ylabel("Frekuensi")
ax.set_title("Distribusi Waktu Pengiriman: Aktual vs Estimasi")
ax.legend()
ax.set_xlim(0, 60)

plt.tight_layout()
plt.show()

print(f"Rata-rata aktual:  {df_kirim['hari_kirim'].mean():.1f} hari")
print(f"Rata-rata estimasi: {df_kirim['hari_estimasi'].mean():.1f} hari")
print(f"Terlambat: {(df_kirim['hari_kirim'] > df_kirim['hari_estimasi']).sum()} order")

In [ ]:
# SQL: berat produk vs ongkir
df_berat = pd.read_sql_query("""
SELECT
    p.product_weight_g AS berat,
    oi.freight_value AS ongkir,
    oi.price AS harga
FROM order_item oi
    JOIN products p ON oi.product_id = p.product_id
""", conn)

# Python: scatter plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(df_berat["berat"], df_berat["ongkir"], alpha=0.1, s=5)
ax.set_xlabel("Berat Produk (g)")
ax.set_ylabel("Ongkos Kirim (BRL)")
ax.set_title("Hubungan Berat Produk vs Ongkos Kirim")
ax.set_xlim(0, 10000)

plt.tight_layout()
plt.show()

---
## 4. Analisis Geografis (Pertanyaan 4)

Distribusi order berdasarkan state dan kota.

In [ ]:
# SQL: jumlah order per state
df_state = pd.read_sql_query("""
SELECT
    c.customer_state AS state,
    COUNT(DISTINCT o.order_id) AS jumlah_order,
    ROUND(SUM(oi.price), 2) AS total_revenue
FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_item oi ON o.order_id = oi.order_id
GROUP BY state
ORDER BY jumlah_order DESC
""", conn)

df_state.head(10)

In [ ]:
# Python: bar chart per state
fig, ax = plt.subplots(figsize=(14, 5))

ax.bar(df_state["state"], df_state["jumlah_order"])
ax.set_xlabel("State")
ax.set_ylabel("Jumlah Order")
ax.set_title("Distribusi Order per State")

plt.tight_layout()
plt.show()

In [ ]:
# SQL: rata-rata waktu kirim per state
df_kirim_state = pd.read_sql_query("""
SELECT
    c.customer_state AS state,
    ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)), 1)
        AS rata_hari_kirim,
    COUNT(*) AS jumlah_order
FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY state
ORDER BY rata_hari_kirim DESC
""", conn)

# Python: bar chart waktu kirim per state
fig, ax = plt.subplots(figsize=(14, 5))

ax.barh(df_kirim_state["state"][::-1], df_kirim_state["rata_hari_kirim"][::-1])
ax.set_xlabel("Rata-rata Hari Pengiriman")
ax.set_ylabel("State")
ax.set_title("Rata-rata Waktu Pengiriman per State")

plt.tight_layout()
plt.show()

---
## 5. Pola Waktu Pembelian (Pertanyaan 5)

Menganalisis pola pembelian berdasarkan hari dalam seminggu dan jam.

In [ ]:
# SQL: ambil timestamp pembelian
df_waktu = pd.read_sql_query("""
SELECT order_purchase_timestamp
FROM orders
""", conn)

# Python: extract hari dan jam
df_waktu["timestamp"] = pd.to_datetime(df_waktu["order_purchase_timestamp"])
df_waktu["hari"] = df_waktu["timestamp"].dt.day_name()
df_waktu["jam"] = df_waktu["timestamp"].dt.hour

# Urutkan hari
hari_urut = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df_waktu["hari"] = pd.Categorical(df_waktu["hari"], categories=hari_urut, ordered=True)

In [ ]:
# Python: heatmap hari x jam
pivot = df_waktu.groupby(["hari", "jam"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, cmap="YlOrRd", annot=False, fmt="d", ax=ax)
ax.set_xlabel("Jam")
ax.set_ylabel("Hari")
ax.set_title("Heatmap Jumlah Order: Hari x Jam")

plt.tight_layout()
plt.show()

In [ ]:
# Python: bar chart per hari
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df_waktu["hari"].value_counts().sort_index().plot(kind="bar", ax=ax1)
ax1.set_xlabel("Hari")
ax1.set_ylabel("Jumlah Order")
ax1.set_title("Jumlah Order per Hari")
ax1.tick_params(axis="x", rotation=45)

df_waktu["jam"].value_counts().sort_index().plot(kind="bar", ax=ax2)
ax2.set_xlabel("Jam")
ax2.set_ylabel("Jumlah Order")
ax2.set_title("Jumlah Order per Jam")

plt.tight_layout()
plt.show()

---
## 6. Korelasi Variabel Numerik

Melihat hubungan antar variabel numerik: harga, ongkir, berat, dan dimensi produk.

In [ ]:
# SQL: gabungkan data numerik
df_num = pd.read_sql_query("""
SELECT
    oi.price AS harga,
    oi.freight_value AS ongkir,
    p.product_weight_g AS berat,
    p.product_length_cm AS panjang,
    p.product_height_cm AS tinggi,
    p.product_width_cm AS lebar
FROM order_item oi
    JOIN products p ON oi.product_id = p.product_id
""", conn)

df_num.describe()

In [ ]:
# Python: heatmap korelasi
corr = df_num.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matriks Korelasi")

plt.tight_layout()
plt.show()

---
## 7. Analisis Jumlah Item per Order (Pertanyaan 5)

Melihat distribusi berapa banyak item dalam satu order.

In [ ]:
# SQL: jumlah item per order
df_item = pd.read_sql_query("""
SELECT
    order_id,
    COUNT(*) AS jumlah_item,
    ROUND(SUM(price), 2) AS total_harga
FROM order_item
GROUP BY order_id
""", conn)

# Python: distribusi
fig, ax = plt.subplots(figsize=(10, 5))

df_item["jumlah_item"].value_counts().sort_index().head(15).plot(kind="bar", ax=ax)
ax.set_xlabel("Jumlah Item per Order")
ax.set_ylabel("Frekuensi")
ax.set_title("Distribusi Jumlah Item per Order")

plt.tight_layout()
plt.show()

print(f"Rata-rata item per order: {df_item['jumlah_item'].mean():.2f}")
print(f"Maks item per order: {df_item['jumlah_item'].max()}")

---
## 8. Analisis RFM (Pertanyaan 5) (Recency, Frequency, Monetary)

Segmentasi pelanggan berdasarkan perilaku pembelian.
Menggunakan `customer_unique_id` karena `customer_id` berbeda setiap order di dataset Olist.

In [ ]:
# Muat data untuk RFM
df_rfm_raw = pd.read_sql_query("""
SELECT
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    oi.price,
    oi.freight_value
FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN order_item oi ON o.order_id = oi.order_id
""", conn)

df_rfm_raw["order_purchase_timestamp"] = pd.to_datetime(df_rfm_raw["order_purchase_timestamp"])
df_rfm_raw["total"] = df_rfm_raw["price"] + df_rfm_raw["freight_value"]

print(f"Jumlah baris: {len(df_rfm_raw)}")
print(f"Customer unik: {df_rfm_raw["customer_unique_id"].nunique()}")

In [ ]:
# Hitung R, F, M per customer
tanggal_referensi = df_rfm_raw["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm = df_rfm_raw.groupby("customer_unique_id").agg(
    recency=("order_purchase_timestamp", lambda x: (tanggal_referensi - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("total", "sum")
).reset_index()

rfm["monetary"] = rfm["monetary"].round(2)

print(f"Tanggal referensi: {tanggal_referensi.date()}")
rfm.describe()

In [ ]:
# Scoring: bagi ke 4 quartile (1-4)
rfm["r_score"] = pd.qcut(rfm["recency"], q=4, labels=[4, 3, 2, 1])
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0, 1, 2, 3, rfm["frequency"].max()], labels=[1, 2, 3, 4], include_lowest=True)
rfm["m_score"] = pd.qcut(rfm["monetary"], q=4, labels=[1, 2, 3, 4])

# Gabungkan skor
rfm["rfm_score"] = rfm["r_score"].astype(str) + rfm["f_score"].astype(str) + rfm["m_score"].astype(str)

rfm.head(10)

In [ ]:
# Segmentasi berdasarkan skor RFM
def segmentasi_rfm(row):
    r, f, m = int(row["r_score"]), int(row["f_score"]), int(row["m_score"])
    if r >= 3 and f >= 3:
        return "Champions"
    elif r >= 3 and f >= 2:
        return "Loyal"
    elif r >= 3:
        return "Potential Loyal"
    elif r >= 2 and f >= 2:
        return "At Risk"
    elif r >= 2:
        return "Need Attention"
    elif f >= 2:
        return "Cant Lose"
    else:
        return "Lost"

rfm["segmen"] = rfm.apply(segmentasi_rfm, axis=1)

print(rfm["segmen"].value_counts())

In [ ]:
# Visualisasi distribusi segmen
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

segmen_count = rfm["segmen"].value_counts()
ax1.barh(segmen_count.index[::-1], segmen_count.values[::-1])
ax1.set_xlabel("Jumlah Customer")
ax1.set_title("Distribusi Segmen RFM")

ax2.pie(segmen_count, labels=segmen_count.index, autopct="%1.1f%%", startangle=90)
ax2.set_title("Proporsi Segmen RFM")

plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata R, F, M per segmen
rfm_summary = rfm.groupby("segmen").agg(
    jumlah=("customer_unique_id", "count"),
    rata_recency=("recency", "mean"),
    rata_frequency=("frequency", "mean"),
    rata_monetary=("monetary", "mean")
).round(1).sort_values("rata_monetary", ascending=False)

rfm_summary

In [ ]:
# Distribusi R, F, M
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rfm["recency"], bins=50, edgecolor="white")
axes[0].set_xlabel("Hari")
axes[0].set_title("Distribusi Recency")

axes[1].hist(rfm["frequency"], bins=range(1, rfm["frequency"].max() + 2), edgecolor="white", align="left")
axes[1].set_xlabel("Jumlah Order")
axes[1].set_title("Distribusi Frequency")

axes[2].hist(rfm["monetary"], bins=50, edgecolor="white")
axes[2].set_xlabel("Total Spending (BRL)")
axes[2].set_title("Distribusi Monetary")
axes[2].set_xlim(0, 2000)

plt.tight_layout()
plt.show()

---
## Penutup

In [ ]:
conn.close()

## Kesimpulan

### Pertanyaan 1: Bagaimana tren pertumbuhan bisnis Olist?
Order tumbuh signifikan dari Sep 2016, mencapai puncak di Nov 2017 (7.186 order, kemungkinan efek Black Friday). Total revenue mencapai 13+ juta BRL. Tren menunjukkan pertumbuhan yang sehat dengan seasonality di akhir tahun.

### Pertanyaan 2: Kategori produk apa yang paling menguntungkan?
Top 5 kategori: cama_mesa_banho, beleza_saude, esporte_lazer, moveis_decoracao, informatica_acessorios. Harga produk sangat skewed (mayoritas di bawah 200 BRL). Peluang untuk fokus stok pada kategori teratas dan segmentasi harga.

### Pertanyaan 3: Apa faktor utama keterlambatan pengiriman?
Rata-rata pengiriman aktual 12,6 hari vs estimasi 23,7 hari (estimasi terlalu konservatif). Namun 8,1% order terlambat. Produk yang terlambat cenderung lebih berat dan lebih mahal. Berat produk berkorelasi sedang dengan ongkir (rho = 0,41).

### Pertanyaan 4: Bagaimana distribusi geografis mempengaruhi pengiriman?
SP mendominasi (40.488 order). State yang jauh dari pusat logistik memiliki waktu kirim dan ongkir yang lebih tinggi. Non-SP memiliki median 12,9 hari vs SP 7,2 hari. Peluang: ekspansi gudang distribusi ke region lain.

### Pertanyaan 5: Bagaimana perilaku pembelian pelanggan?
Peak pembelian di jam 16:00 hari Senin. Weekend hanya 23% transaksi. Rata-rata 1,14 item per order (peluang cross-selling). Mayoritas pelanggan adalah one-time buyer berdasarkan analisis RFM. Fokus retensi: program loyalitas dan re-engagement campaign.